# BridgeLink ASL - WLASL-100 Training (Google Colab)

End-to-end pipeline: use your uploaded `wlasl-processed.zip` from Google Drive, extract MediaPipe Holistic landmarks, train a lightweight Transformer classifier, evaluate WLASL-100, and save model/report artifacts.

**Before running:**
1. In Colab, choose `Runtime -> Change runtime type -> GPU`.
2. Upload/place `wlasl-processed.zip` at `My Drive/BridgeLink-ASL/data/wlasl-processed.zip`.
3. Run the notebook top-to-bottom. The unzip step is skipped automatically after the dataset has been extracted once.
4. Outputs are saved to Google Drive under `BridgeLink-ASL/` so they survive runtime resets.


## 1. Setup

In [ ]:
# Stable setup for training/evaluation.
# Do NOT install/downgrade MediaPipe here. Colab can break NumPy when MediaPipe
# changes binary dependencies inside an already-running session.
# MediaPipe is only needed if you rerun landmark extraction in Step 7.

import json
import math
import os
import random
import zipfile
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
print("NumPy version:", np.__version__)
print("Torch version:", torch.__version__)


## 2. Mount Google Drive

All outputs are saved to Drive so you don't lose them if Colab disconnects.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Project folder on your Drive.
PROJECT_DIR = Path("/content/drive/MyDrive/BridgeLink-ASL")
DATA_DIR = PROJECT_DIR / "data"
DATA_ZIP = DATA_DIR / "wlasl-processed.zip"
EXTRACTED_DIR = DATA_DIR / "wlasl-processed"

# Some Google Drive zips extract directly into data/ while others extract into
# data/wlasl-processed/. Support both layouts automatically.
if (DATA_DIR / "videos").exists() or (DATA_DIR / "WLASL_v0.3.json").exists():
    DATASET_DIR = DATA_DIR
else:
    DATASET_DIR = EXTRACTED_DIR

# Output folders
PROJECT_DIR.mkdir(exist_ok=True, parents=True)
DATA_DIR.mkdir(exist_ok=True, parents=True)
(PROJECT_DIR / "models").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
(PROJECT_DIR / "landmarks").mkdir(exist_ok=True)

# Optional local scratch directory. The dataset itself is read from Drive.
LOCAL = Path("/content/wlasl")
LOCAL.mkdir(exist_ok=True)

print("Drive project dir:", PROJECT_DIR)
print("Dataset zip     :", DATA_ZIP)
print("Dataset dir     :", DATASET_DIR)
print("Zip exists      :", DATA_ZIP.exists())


## 3. Unzip the uploaded WLASL dataset from Google Drive

This notebook expects your uploaded file here:

`/content/drive/MyDrive/BridgeLink-ASL/data/wlasl-processed.zip`

The zip is extracted to:

`/content/drive/MyDrive/BridgeLink-ASL/data/wlasl-processed/`

If that folder already contains files, the notebook skips extraction so you do not redo the slow step every run.


In [ ]:
if not DATA_ZIP.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_ZIP}. Make sure the zip is in My Drive/BridgeLink-ASL/data/"
    )

print("Found uploaded dataset zip.")
print(f"Size: {DATA_ZIP.stat().st_size / 1e9:.2f} GB")


In [ ]:
import zipfile

if DATASET_DIR == DATA_DIR:
    print("Dataset appears to be extracted directly under data/. Skipping extraction.")
else:
    needs_extract = (not DATASET_DIR.exists()) or (not any(DATASET_DIR.iterdir()))

    if needs_extract:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(f"Could not find dataset zip: {DATA_ZIP}")
        print("Unzipping dataset to Google Drive. This can take several minutes...")
        DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP, "r") as z:
            z.extractall(DATASET_DIR)
        print("Unzip complete.")
    else:
        print("Dataset already unzipped. Skipping extraction.")

print("Top-level dataset contents:")
for p in list(DATASET_DIR.iterdir())[:30]:
    print(" -", p.name)


## 4. Locate the dataset files

In [ ]:
from pathlib import Path

def find_wlasl_assets(root):
    root = Path(root)
    json_candidates = list(root.rglob("WLASL_v0.3.json")) + list(root.rglob("wlasl_v0.3.json"))
    if not json_candidates:
        # Try case-insensitive
        json_candidates = [p for p in root.rglob("*.json") if "wlasl" in p.name.lower() and "v0" in p.name.lower()]
    if not json_candidates:
        raise FileNotFoundError(f"WLASL JSON not found under {root}. Check download.")
    json_path = json_candidates[0]
    for candidate in [json_path.parent / "videos", json_path.parent, json_path.parent.parent / "videos"]:
        if candidate.exists() and any(candidate.glob("*.mp4")):
            return json_path, candidate
    for d in root.rglob("*"):
        if d.is_dir() and any(d.glob("*.mp4")):
            return json_path, d
    raise FileNotFoundError("Could not locate WLASL mp4 videos.")

JSON_PATH, VIDEO_DIR = find_wlasl_assets(DATASET_DIR)
print("JSON :", JSON_PATH)
print("Videos:", VIDEO_DIR)
num_videos = len(list(VIDEO_DIR.glob('*.mp4')))
print(f"Total mp4 files: {num_videos}")
print("Sample:", [p.name for p in list(VIDEO_DIR.glob('*.mp4'))[:5]])


## 5. Parse WLASL JSON → filter to WLASL-100

In [ ]:
import json
import random
from collections import Counter
import numpy as np

with open(JSON_PATH) as f:
    raw = json.load(f)

all_samples = []
for entry in raw:
    gloss = entry["gloss"]
    for inst in entry["instances"]:
        # Filter to WLASL-100 subset
        subsets = inst.get("subsets", [])
        # Some mirrors use different key names — handle both
        if not subsets:
            subsets = inst.get("subset", [])
        if isinstance(subsets, str):
            subsets = [subsets]
        if "WLASL100" not in subsets and "asl100" not in [s.lower() for s in subsets]:
            continue
        video_id = inst["video_id"]
        video_path = VIDEO_DIR / f"{video_id}.mp4"
        if not video_path.exists():
            continue
        all_samples.append({
            "gloss": gloss,
            "video_id": video_id,
            "video_path": str(video_path),
            "split": inst.get("split", "train"),
            "frame_start": inst.get("frame_start", 1),
            "frame_end": inst.get("frame_end", -1),
            "bbox": inst.get("bbox", None),
        })

print(f"WLASL-100 instances with available videos: {len(all_samples)}")

# FALLBACK: if subset tags are missing, take the top-100 glosses by frequency
if len(all_samples) < 50:
    print("WARNING: subset tags not found — falling back to top-100 glosses by frequency.")
    all_samples_full = []
    for entry in raw:
        gloss = entry["gloss"]
        for inst in entry["instances"]:
            video_id = inst["video_id"]
            video_path = VIDEO_DIR / f"{video_id}.mp4"
            if not video_path.exists():
                continue
            all_samples_full.append({
                "gloss": gloss,
                "video_id": video_id,
                "video_path": str(video_path),
                "split": inst.get("split", "train"),
                "frame_start": inst.get("frame_start", 1),
                "frame_end": inst.get("frame_end", -1),
                "bbox": inst.get("bbox", None),
            })
    # Keep top 100 glosses
    gloss_counts = Counter(s["gloss"] for s in all_samples_full)
    top100 = {g for g, _ in gloss_counts.most_common(100)}
    all_samples = [s for s in all_samples_full if s["gloss"] in top100]
    print(f"After top-100 fallback: {len(all_samples)} instances")

split_counts = Counter(s["split"] for s in all_samples)
class_counts = Counter(s["gloss"] for s in all_samples)
print(f"Split distribution: {dict(split_counts)}")
print(f"Classes present: {len(class_counts)}")
print(f"Median samples/class: {int(np.median(list(class_counts.values())))}")

# If no split info, create random 80/10/10
if len(split_counts) <= 1:
    print("No split info found — creating random 80/10/10 split.")
    random.shuffle(all_samples)
    n = len(all_samples)
    for i, s in enumerate(all_samples):
        if i < int(0.8 * n):
            s["split"] = "train"
        elif i < int(0.9 * n):
            s["split"] = "val"
        else:
            s["split"] = "test"
    split_counts = Counter(s["split"] for s in all_samples)
    print(f"New split distribution: {dict(split_counts)}")


## 6. Dataset visualizations

In [ ]:
import matplotlib.pyplot as plt

RESULTS = PROJECT_DIR / "results"

# Class distribution
top_classes = class_counts.most_common(30)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([c[0] for c in top_classes], [c[1] for c in top_classes], color="#4c72b0")
ax.set_title("WLASL-100 — top 30 classes by instance count")
ax.set_ylabel("# clips")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.savefig(RESULTS / "class_distribution.png", dpi=150)
plt.show()

# Split distribution
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(split_counts.keys(), split_counts.values(), color=["#4c72b0", "#55a868", "#c44e52"])
ax.set_title("Train / val / test split")
ax.set_ylabel("# clips")
plt.tight_layout()
plt.savefig(RESULTS / "split_distribution.png", dpi=150)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

# Preview sample frames
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, sample in zip(axes, random.sample(all_samples, min(5, len(all_samples)))):
    cap = cv2.VideoCapture(sample["video_path"])
    ok, frame = cap.read()
    cap.release()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(sample["gloss"], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS / "sample_frames.png", dpi=150)
plt.show()


## 7. MediaPipe Holistic landmark extraction

Converts each clip into a `(32, 225)` float32 array.

If `BridgeLink-ASL/landmarks/` and `landmarks_manifest.json` already exist in
Google Drive, you can skip this section and continue to the dataset/CNN cells.

Only run the install cell below if you truly need to extract landmarks again.
After installing MediaPipe, use `Runtime -> Restart session`, then run from the
top. This avoids NumPy binary-compatibility errors in Colab.


In [ ]:
# OPTIONAL: only run this if you need to re-extract landmarks.
# After this cell finishes, immediately use Runtime -> Restart session.
!pip uninstall -y -q mediapipe
!pip install -q mediapipe==0.10.21
print("MediaPipe installed. Now use Runtime -> Restart session, then run from the top.")


In [ ]:
import cv2
import numpy as np

try:
    import mediapipe as mp
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "MediaPipe is not installed. If landmarks are already cached, skip this "
        "section and continue to the dataset/CNN cells. If you need extraction, "
        "run the optional MediaPipe install cell, restart the session, then run again."
    ) from exc

print("MediaPipe version:", getattr(mp, "__version__", "unknown"))
print("MediaPipe module :", getattr(mp, "__file__", "unknown"))

if not hasattr(mp, "solutions"):
    raise RuntimeError(
        "This MediaPipe build does not expose mp.solutions. Run the optional "
        "MediaPipe install cell, then Runtime -> Restart session, then run from the top."
    )

mp_holistic = mp.solutions.holistic
print("Using MediaPipe import: mp.solutions.holistic")

SEQ_LEN = 32
FEAT_DIM = 21*3 + 21*3 + 33*3  # 225

def flatten_lm(landmarks, n):
    if landmarks is None:
        return np.zeros(n * 3, dtype=np.float32)
    return np.array([[p.x, p.y, p.z] for p in landmarks.landmark], dtype=np.float32).flatten()

def extract_clip(video_path, frame_start, frame_end, bbox=None, seq_len=SEQ_LEN):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return None
    start = max(0, frame_start - 1)
    end = total if frame_end == -1 else min(total, frame_end)
    end = max(start + 1, end)
    frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    for _ in range(end - start):
        ok, frame = cap.read()
        if not ok:
            break
        if bbox is not None:
            x1, y1, x2, y2 = [int(v) for v in bbox]
            h, w = frame.shape[:2]
            x1, y1, x2, y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
            if x2 > x1 and y2 > y1:
                frame = frame[y1:y2, x1:x2]
        frames.append(frame)
    cap.release()
    if not frames:
        return None
    idx = np.linspace(0, len(frames) - 1, seq_len).astype(int)
    sampled = [frames[i] for i in idx]
    seq = np.zeros((seq_len, FEAT_DIM), dtype=np.float32)
    with mp_holistic.Holistic(static_image_mode=False, model_complexity=1,
                              min_detection_confidence=0.3, min_tracking_confidence=0.3) as holistic:
        for t, f in enumerate(sampled):
            rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            res = holistic.process(rgb)
            lh = flatten_lm(res.left_hand_landmarks, 21)
            rh = flatten_lm(res.right_hand_landmarks, 21)
            pose = flatten_lm(res.pose_landmarks, 33)
            seq[t] = np.concatenate([lh, rh, pose])
    return seq


In [ ]:
import json
import numpy as np
from tqdm.auto import tqdm

LANDMARK_DIR = PROJECT_DIR / "landmarks"
LANDMARK_DIR.mkdir(exist_ok=True)

manifest_path = PROJECT_DIR / "landmarks_manifest.json"

# If everything is already cached, load the manifest and skip the slow pass.
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    cached_files = sum(1 for m in manifest if Path(m.get("landmark_path", "")).exists())
    if cached_files == len(manifest) and len(manifest) > 0:
        print(f"Loaded cached landmark manifest: {len(manifest)} samples")
        print("Skipping MediaPipe extraction because all cached .npy files exist.")
    else:
        print(f"Manifest exists, but only {cached_files}/{len(manifest)} landmark files were found.")

if "manifest" not in globals() or not manifest:
    if "extract_clip" not in globals():
        raise RuntimeError(
            "Landmarks are not cached and extract_clip is unavailable. Run the optional "
            "MediaPipe install/import cells first, restart if needed, then rerun extraction."
        )

    cached = extracted = failed = 0
    manifest = []

    for sample in tqdm(all_samples, desc="Extracting landmarks"):
        out_path = LANDMARK_DIR / f"{sample['video_id']}.npy"
        if out_path.exists():
            cached += 1
            manifest.append({**sample, "landmark_path": str(out_path)})
            continue
        try:
            seq = extract_clip(sample["video_path"], sample["frame_start"],
                               sample["frame_end"], sample.get("bbox"))
            if seq is None:
                failed += 1
                continue
            np.save(out_path, seq)
            manifest.append({**sample, "landmark_path": str(out_path)})
            extracted += 1
        except Exception as e:
            failed += 1
            if failed <= 5:
                print(f"  fail {sample['video_id']}: {e}")

    print(f"\nExtracted: {extracted}  Cached: {cached}  Failed: {failed}")
    print(f"Usable samples: {len(manifest)}")

    with open(manifest_path, "w") as f:
        json.dump(manifest, f)


## 8. PyTorch dataset with augmentation

In [ ]:
import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

if 'manifest' not in globals():
    manifest_path = PROJECT_DIR / 'landmarks_manifest.json'
    if not manifest_path.exists():
        raise RuntimeError('No manifest in memory and landmarks_manifest.json was not found. Run landmark extraction first.')
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f'Loaded cached landmark manifest: {len(manifest)} samples')

glosses = sorted({m['gloss'] for m in manifest})
label_map = {g: i for i, g in enumerate(glosses)}
inv_label_map = {i: g for g, i in label_map.items()}
print(f"Classes: {len(label_map)}")

train_samples = [m for m in manifest if m['split'] == 'train']
val_samples   = [m for m in manifest if m['split'] == 'val']
test_samples  = [m for m in manifest if m['split'] == 'test']
print(f"Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}")

class LandmarkSignDataset(Dataset):
    def __init__(self, samples, label_map, augment=False):
        self.samples = samples
        self.label_map = label_map
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        x = np.load(s['landmark_path'])
        y = self.label_map[s['gloss']]
        if self.augment:
            x = self._augment(x)
        return torch.from_numpy(x.astype(np.float32)), y

    def _augment(self, x):
        x = x + np.random.normal(0, 0.01, x.shape).astype(np.float32)
        if random.random() < 0.5:
            x = x.copy()
            x[:, 0::3] = 1.0 - x[:, 0::3]
            lh = x[:, 0:63].copy()
            rh = x[:, 63:126].copy()
            x[:, 0:63] = rh
            x[:, 63:126] = lh
        if random.random() < 0.3:
            start = random.randint(0, max(0, x.shape[0] - 4))
            x = x.copy()
            x[start:start+3] = 0.0
        return x

train_ds = LandmarkSignDataset(train_samples, label_map, augment=True)
val_ds   = LandmarkSignDataset(val_samples, label_map, augment=False)
test_ds  = LandmarkSignDataset(test_samples, label_map, augment=False)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)
print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")


## 9. CNN baseline model for the required CNN vs VLM comparison

This is the primary model for the professor-facing comparison. It uses the same cached MediaPipe landmark tensors, but replaces attention with a temporal 1D CNN over the 32-frame sequence. The Transformer section below is kept as an optional/extra-credit attention model.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

class LandmarkCNN(nn.Module):
    def __init__(self, num_classes, feat_dim=225, channels=(128, 128, 256), dropout=0.35):
        super().__init__()
        c1, c2, c3 = channels
        self.features = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Conv1d(feat_dim, c1, kernel_size=3, padding=1),
            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(c2, c3, kernel_size=3, padding=1),
            nn.BatchNorm1d(c3),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(c3, num_classes)
        self.config = {
            "model_type": "landmark_cnn",
            "num_classes": num_classes,
            "feat_dim": feat_dim,
            "channels": list(channels),
            "dropout": dropout,
        }

    def forward(self, x):
        # x: (batch, time, features); Conv1d expects (batch, features, time)
        h = x.transpose(1, 2)
        h = self.features(h).squeeze(-1)
        return self.head(h)


def evaluate_cnn(loader, model_obj):
    model_obj.eval()
    correct1 = correct5 = total = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model_obj(x)
            top5 = logits.topk(min(5, logits.size(1)), dim=1).indices
            pred = logits.argmax(dim=1)
            correct1 += (pred == y).sum().item()
            correct5 += (top5 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
            all_preds.extend(pred.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return correct1 / max(total, 1), correct5 / max(total, 1), all_preds, all_labels

cnn_model = LandmarkCNN(num_classes=len(label_map)).to(DEVICE)
cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"CNN params: {cnn_params/1e6:.2f}M")

CNN_EPOCHS = 50
CNN_LR = 1e-3
CNN_WEIGHT_DECAY = 1e-2

cnn_optimizer = AdamW(cnn_model.parameters(), lr=CNN_LR, weight_decay=CNN_WEIGHT_DECAY)
cnn_scheduler = CosineAnnealingLR(cnn_optimizer, T_max=CNN_EPOCHS)
cnn_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

cnn_history = {"train_loss": [], "val_acc": [], "val_top5": []}
cnn_best_val = -1.0
cnn_best_epoch = -1
cnn_best_path = PROJECT_DIR / "models/cnn_landmark_best.pt"

print("Training CNN baseline...")
for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = cnn_model(x)
        loss = cnn_loss_fn(logits, y)
        cnn_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)
        cnn_optimizer.step()
        running += loss.item() * y.size(0)
    cnn_scheduler.step()

    train_loss = running / max(len(train_ds), 1)
    val_top1, val_top5, _, _ = evaluate_cnn(val_loader, cnn_model)
    cnn_history["train_loss"].append(train_loss)
    cnn_history["val_acc"].append(val_top1)
    cnn_history["val_top5"].append(val_top5)
    marker = ""
    if val_top1 > cnn_best_val:
        cnn_best_val = val_top1
        cnn_best_epoch = epoch
        torch.save({
            "state_dict": cnn_model.state_dict(),
            "label_map": label_map,
            "config": cnn_model.config,
        }, cnn_best_path)
        marker = " <- best"
    if epoch % 5 == 0 or marker:
        print(f"cnn epoch {epoch:02d}  loss={train_loss:.3f}  val_top1={val_top1:.3f}  val_top5={val_top5:.3f}{marker}")

print(f"\nBest CNN val top-1: {cnn_best_val:.3f} at epoch {cnn_best_epoch}")

ckpt = torch.load(cnn_best_path, map_location=DEVICE, weights_only=False)
cnn_model.load_state_dict(ckpt["state_dict"])
cnn_test_top1, cnn_test_top5, cnn_preds, cnn_labels = evaluate_cnn(test_loader, cnn_model)
print(f"CNN TEST  top-1: {cnn_test_top1:.3f}   top-5: {cnn_test_top5:.3f}")

try:
    from sklearn.metrics import classification_report, confusion_matrix
    present_labels = sorted(set(cnn_labels + cnn_preds))
    target_names = [inv_label_map[i] for i in present_labels]
    cnn_report_txt = classification_report(
        cnn_labels,
        cnn_preds,
        labels=present_labels,
        target_names=target_names,
        zero_division=0,
        digits=3,
    )
    print(cnn_report_txt)
    with open(PROJECT_DIR / "results/cnn_classification_report.txt", "w") as f:
        f.write(cnn_report_txt)
    cnn_cm = confusion_matrix(cnn_labels, cnn_preds, labels=list(range(len(label_map))))
except ImportError:
    cnn_cm = np.zeros((len(label_map), len(label_map)), dtype=int)
    for p, l in zip(cnn_preds, cnn_labels):
        cnn_cm[l, p] += 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cnn_history["train_loss"])
axes[0].set_title("CNN training loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[1].plot(cnn_history["val_acc"], label="top-1")
axes[1].plot(cnn_history["val_top5"], label="top-5")
axes[1].set_title("CNN validation accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/cnn_training_curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cnn_cm, cmap="Blues")
ax.set_title(f"CNN confusion matrix (test) - top-1 {cnn_test_top1:.1%}")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/cnn_confusion_matrix.png", dpi=150)
plt.show()

cnn_metrics = {
    "model": "landmark_cnn",
    "val_top1_best": round(cnn_best_val, 4),
    "val_top1_best_epoch": cnn_best_epoch,
    "test_top1": round(cnn_test_top1, 4),
    "test_top5": round(cnn_test_top5, 4),
    "num_classes": len(label_map),
    "train_samples": len(train_samples),
    "val_samples": len(val_samples),
    "test_samples": len(test_samples),
    "model_params_M": round(cnn_params / 1e6, 2),
    "epochs": CNN_EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": CNN_LR,
    "weight_decay": CNN_WEIGHT_DECAY,
    "loss": "cross_entropy_label_smoothing_0.1",
}
with open(PROJECT_DIR / "results/cnn_metrics.json", "w") as f:
    json.dump(cnn_metrics, f, indent=2)
print(json.dumps(cnn_metrics, indent=2))


## 10. Train WLASL-25 CNN for the live demo

The WLASL-100 CNN is the report-scale experiment. This smaller WLASL-25 CNN uses the same cached landmarks but restricts the output vocabulary to the 25 most frequent signs, which should make the live Hugging Face demo much more stable.


In [ ]:
import json
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
DEMO_CLASSES = 25

class_counts = Counter(m["gloss"] for m in manifest)
demo_glosses = [g for g, _ in class_counts.most_common(DEMO_CLASSES)]
demo_gloss_set = set(demo_glosses)
print("Demo classes:", demo_glosses)

# Reuse the original split labels, but only keep samples from the demo vocabulary.
demo_train_samples = [m for m in train_samples if m["gloss"] in demo_gloss_set]
demo_val_samples = [m for m in val_samples if m["gloss"] in demo_gloss_set]
demo_test_samples = [m for m in test_samples if m["gloss"] in demo_gloss_set]

# If a class has no official test sample after filtering, the model still trains;
# the report should mention the exact held-out demo sample count below.
demo_label_map = {g: i for i, g in enumerate(demo_glosses)}
demo_inv_label_map = {i: g for g, i in demo_label_map.items()}

print(f"Demo train/val/test: {len(demo_train_samples)} / {len(demo_val_samples)} / {len(demo_test_samples)}")
print(f"Demo classes with train samples: {len(set(m['gloss'] for m in demo_train_samples))}")
print(f"Demo classes with test samples : {len(set(m['gloss'] for m in demo_test_samples))}")

class DemoLandmarkSignDataset(LandmarkSignDataset):
    def __init__(self, samples, label_map, augment=False):
        super().__init__(samples, label_map, augment=augment)

DEMO_BATCH = 32
demo_train_ds = DemoLandmarkSignDataset(demo_train_samples, demo_label_map, augment=True)
demo_val_ds = DemoLandmarkSignDataset(demo_val_samples, demo_label_map, augment=False)
demo_test_ds = DemoLandmarkSignDataset(demo_test_samples, demo_label_map, augment=False)

demo_train_loader = DataLoader(demo_train_ds, batch_size=DEMO_BATCH, shuffle=True, num_workers=2)
demo_val_loader = DataLoader(demo_val_ds, batch_size=DEMO_BATCH, shuffle=False, num_workers=2)
demo_test_loader = DataLoader(demo_test_ds, batch_size=DEMO_BATCH, shuffle=False, num_workers=2)

# Reuse the same CNN class, but with 25 output classes.
demo_cnn_model = LandmarkCNN(num_classes=len(demo_label_map), channels=(128, 128, 256), dropout=0.30).to(DEVICE)
demo_params = sum(p.numel() for p in demo_cnn_model.parameters())
print(f"Demo CNN params: {demo_params/1e6:.2f}M")

DEMO_EPOCHS = 60
DEMO_LR = 1e-3
DEMO_WEIGHT_DECAY = 1e-2

demo_optimizer = AdamW(demo_cnn_model.parameters(), lr=DEMO_LR, weight_decay=DEMO_WEIGHT_DECAY)
demo_scheduler = CosineAnnealingLR(demo_optimizer, T_max=DEMO_EPOCHS)
demo_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

demo_history = {"train_loss": [], "val_acc": [], "val_top5": []}
demo_best_val = -1.0
demo_best_epoch = -1
demo_best_path = PROJECT_DIR / "models/cnn_landmark_wlasl25_best.pt"

def evaluate_demo(loader, model_obj):
    model_obj.eval()
    correct1 = correct5 = total = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model_obj(x)
            top5 = logits.topk(min(5, logits.size(1)), dim=1).indices
            pred = logits.argmax(dim=1)
            correct1 += (pred == y).sum().item()
            correct5 += (top5 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
            all_preds.extend(pred.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return correct1 / max(total, 1), correct5 / max(total, 1), all_preds, all_labels

print("Training WLASL-25 demo CNN...")
for epoch in range(DEMO_EPOCHS):
    demo_cnn_model.train()
    running = 0.0
    for x, y in demo_train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = demo_cnn_model(x)
        loss = demo_loss_fn(logits, y)
        demo_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(demo_cnn_model.parameters(), 1.0)
        demo_optimizer.step()
        running += loss.item() * y.size(0)
    demo_scheduler.step()

    train_loss = running / max(len(demo_train_ds), 1)
    val_top1, val_top5, _, _ = evaluate_demo(demo_val_loader, demo_cnn_model)
    demo_history["train_loss"].append(train_loss)
    demo_history["val_acc"].append(val_top1)
    demo_history["val_top5"].append(val_top5)
    marker = ""
    if val_top1 > demo_best_val:
        demo_best_val = val_top1
        demo_best_epoch = epoch
        torch.save({
            "state_dict": demo_cnn_model.state_dict(),
            "label_map": demo_label_map,
            "config": demo_cnn_model.config,
        }, demo_best_path)
        marker = " <- best"
    if epoch % 5 == 0 or marker:
        print(f"demo epoch {epoch:02d}  loss={train_loss:.3f}  val_top1={val_top1:.3f}  val_top5={val_top5:.3f}{marker}")

print(f"\nBest demo val top-1: {demo_best_val:.3f} at epoch {demo_best_epoch}")

ckpt = torch.load(demo_best_path, map_location=DEVICE, weights_only=False)
demo_cnn_model.load_state_dict(ckpt["state_dict"])
demo_test_top1, demo_test_top5, demo_preds, demo_labels = evaluate_demo(demo_test_loader, demo_cnn_model)
print(f"DEMO CNN TEST  top-1: {demo_test_top1:.3f}   top-5: {demo_test_top5:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(demo_history["train_loss"])
axes[0].set_title("WLASL-25 demo CNN training loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[1].plot(demo_history["val_acc"], label="top-1")
axes[1].plot(demo_history["val_top5"], label="top-5")
axes[1].set_title("WLASL-25 demo CNN validation accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/demo_wlasl25_training_curves.png", dpi=150)
plt.show()

try:
    from sklearn.metrics import classification_report, confusion_matrix
    present_labels = sorted(set(demo_labels + demo_preds))
    target_names = [demo_inv_label_map[i] for i in present_labels]
    demo_report_txt = classification_report(
        demo_labels,
        demo_preds,
        labels=present_labels,
        target_names=target_names,
        zero_division=0,
        digits=3,
    )
    print(demo_report_txt)
    with open(PROJECT_DIR / "results/demo_wlasl25_classification_report.txt", "w") as f:
        f.write(demo_report_txt)
    demo_cm = confusion_matrix(demo_labels, demo_preds, labels=list(range(len(demo_label_map))))
except ImportError:
    demo_cm = np.zeros((len(demo_label_map), len(demo_label_map)), dtype=int)
    for p, l in zip(demo_preds, demo_labels):
        demo_cm[l, p] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(demo_cm, cmap="Blues")
ax.set_title(f"WLASL-25 demo CNN confusion matrix - top-1 {demo_test_top1:.1%}")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/demo_wlasl25_confusion_matrix.png", dpi=150)
plt.show()

demo_metrics = {
    "model": "landmark_cnn_wlasl25_demo",
    "val_top1_best": round(demo_best_val, 4),
    "val_top1_best_epoch": demo_best_epoch,
    "test_top1": round(demo_test_top1, 4),
    "test_top5": round(demo_test_top5, 4),
    "num_classes": len(demo_label_map),
    "classes": demo_glosses,
    "train_samples": len(demo_train_samples),
    "val_samples": len(demo_val_samples),
    "test_samples": len(demo_test_samples),
    "model_params_M": round(demo_params / 1e6, 2),
    "epochs": DEMO_EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": DEMO_LR,
    "weight_decay": DEMO_WEIGHT_DECAY,
    "loss": "cross_entropy_label_smoothing_0.1",
}
with open(PROJECT_DIR / "results/demo_wlasl25_metrics.json", "w") as f:
    json.dump(demo_metrics, f, indent=2)
print(json.dumps(demo_metrics, indent=2))


## 10. Build the CNN-top-5 dataset for VLM reranking

This creates the small held-out evaluation set used for the final CNN vs VLM comparison. The VLM should choose from the CNN's top-5 candidate labels only, then we compare CNN top-1, CNN top-5 coverage, and VLM-reranked accuracy.


In [ ]:
import json
import shutil
from collections import Counter
from pathlib import Path

HYBRID_DIR = PROJECT_DIR / "vlm_eval_wlasl25_cnn"
CLIP_DIR = HYBRID_DIR / "clips"
HYBRID_DIR.mkdir(exist_ok=True)
CLIP_DIR.mkdir(exist_ok=True)

class_counts = Counter(m["gloss"] for m in manifest)
top25 = {g for g, _ in class_counts.most_common(25)}
hybrid_samples = [m for m in test_samples if m["gloss"] in top25]

print("CNN/VLM hybrid eval samples:", len(hybrid_samples))
print("Classes:", len(set(m["gloss"] for m in hybrid_samples)))

ckpt = torch.load(cnn_best_path, map_location=DEVICE, weights_only=False)
cnn_model.load_state_dict(ckpt["state_dict"])
cnn_model.eval()

rows = []
with torch.no_grad():
    for sample in hybrid_samples:
        x = np.load(sample["landmark_path"]).astype(np.float32)
        xt = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
        logits = cnn_model(xt)
        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
        top5_idx = probs.argsort()[::-1][:5]
        top5 = [
            {"label": inv_label_map[int(i)], "confidence": float(probs[int(i)])}
            for i in top5_idx
        ]

        src_video = Path(sample["video_path"])
        dst_video = CLIP_DIR / f"{sample['video_id']}_{sample['gloss']}.mp4"
        if src_video.exists() and not dst_video.exists():
            shutil.copy2(src_video, dst_video)

        rows.append({
            "candidate_model": "landmark_cnn",
            "video_id": sample["video_id"],
            "true_label": sample["gloss"],
            "video_path": str(dst_video),
            "landmark_path": sample["landmark_path"],
            "cnn_top1": top5[0]["label"],
            "cnn_top1_confidence": top5[0]["confidence"],
            "cnn_top5": top5,
            "model_top1": top5[0]["label"],
            "model_top1_confidence": top5[0]["confidence"],
            "model_top5": top5,
            "vlm_prompt": (
                "You are classifying an isolated ASL sign from a short video. "
                "Choose the best matching label from this candidate list only: "
                f"{[c['label'] for c in top5]}. "
                "Return only the chosen label and one short reason."
            ),
        })

jsonl_path = HYBRID_DIR / "wlasl25_cnn_hybrid_eval.jsonl"
with open(jsonl_path, "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

print("Saved:", jsonl_path)
print("Saved clips to:", CLIP_DIR)


## 11. Optional Sign Transformer model (~1.3M params, extra attention experiment)

In [ ]:
import torch
import torch.nn as nn

DEVICE = globals().get('DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

class SignTransformer(nn.Module):
    def __init__(self, num_classes, d_model=192, nhead=4, layers=4,
                 seq_len=32, feat_dim=225, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len + 1, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self.config = dict(num_classes=num_classes, d_model=d_model, nhead=nhead,
                           layers=layers, seq_len=seq_len, feat_dim=feat_dim, dropout=dropout)

    def forward(self, x):
        B = x.size(0)
        h = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1) + self.pos_embed
        h = self.encoder(h)
        h = self.norm(h[:, 0])
        return self.head(h)

model = SignTransformer(num_classes=len(label_map)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.2f}M")


## 12. Optional Transformer training loop

Best model is saved to Google Drive after every improvement.

In [ ]:
import json
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = globals().get('DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')

EPOCHS = 60
LR = 3e-4
WEIGHT_DECAY = 1e-2

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

history = {"train_loss": [], "val_acc": [], "val_top5": []}
best_val = 0.0
best_epoch = -1
best_path = PROJECT_DIR / "models/sign_transformer_best.pt"

def evaluate(loader):
    model.eval()
    correct1 = correct5 = total = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            top5 = logits.topk(min(5, logits.size(1)), dim=1).indices
            pred = logits.argmax(dim=1)
            correct1 += (pred == y).sum().item()
            correct5 += (top5 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
            all_preds.extend(pred.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return correct1/max(total,1), correct5/max(total,1), all_preds, all_labels

print("Training...")
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item() * y.size(0)
    scheduler.step()
    train_loss = running / max(len(train_ds), 1)

    val_top1, val_top5, _, _ = evaluate(val_loader)
    history["train_loss"].append(train_loss)
    history["val_acc"].append(val_top1)
    history["val_top5"].append(val_top5)
    marker = ""
    if val_top1 > best_val:
        best_val = val_top1
        best_epoch = epoch
        torch.save({
            "state_dict": model.state_dict(),
            "label_map": label_map,
            "config": model.config,
        }, best_path)
        marker = "  ← best"
    if epoch % 5 == 0 or marker:
        print(f"epoch {epoch:02d}  loss={train_loss:.3f}  val_top1={val_top1:.3f}  val_top5={val_top5:.3f}{marker}")

print(f"\nBest val top-1: {best_val:.3f} at epoch {best_epoch}")


## 13. Optional Transformer training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[1].plot(history["val_acc"], label="top-1")
axes[1].plot(history["val_top5"], label="top-5")
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/training_curves.png", dpi=150)
plt.show()


## 14. Optional Transformer final test evaluation

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch

DEVICE = globals().get('DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])

test_top1, test_top5, preds, labels = evaluate(test_loader)
print(f"TEST  top-1: {test_top1:.3f}   top-5: {test_top5:.3f}")

try:
    from sklearn.metrics import classification_report, confusion_matrix
    present_labels = sorted(set(labels + preds))
    target_names = [inv_label_map[i] for i in present_labels]
    report_txt = classification_report(labels, preds, labels=present_labels,
                                       target_names=target_names, zero_division=0, digits=3)
    print(report_txt)
    with open(PROJECT_DIR / "results/classification_report.txt", "w") as f:
        f.write(report_txt)
    cm = confusion_matrix(labels, preds, labels=list(range(len(label_map))))
except ImportError:
    print("sklearn not found — computing confusion matrix manually")
    cm = np.zeros((len(label_map), len(label_map)), dtype=int)
    for p, l in zip(preds, labels):
        cm[l, p] += 1

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, cmap="Blues")
ax.set_title(f"Confusion matrix (test) — top-1 {test_top1:.1%}")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/confusion_matrix.png", dpi=150)
plt.show()

metrics = {
    "val_top1_best": round(best_val, 4),
    "val_top1_best_epoch": best_epoch,
    "test_top1": round(test_top1, 4),
    "test_top5": round(test_top5, 4),
    "num_classes": len(label_map),
    "train_samples": len(train_samples),
    "val_samples": len(val_samples),
    "test_samples": len(test_samples),
    "model_params_M": round(n_params / 1e6, 2),
    "epochs": EPOCHS,
}
with open(PROJECT_DIR / "results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


## 15. Export

In [ ]:
import json

with open(PROJECT_DIR / "models/labels.json", "w") as f:
    json.dump({
        "label_map": label_map,
        "inv_label_map": {str(i): g for i, g in inv_label_map.items()}
    }, f, indent=2)

print()
print("Saved to Google Drive at:", PROJECT_DIR)
print()
print("Models:")
for p in sorted((PROJECT_DIR / "models").iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.2f} MB)")
print()
print("Results:")
for p in sorted((PROJECT_DIR / "results").iterdir()):
    print(f"  {p.name}")
print()
print("=" * 60)
print("DONE! Next steps:")
print("1. Go to Google Drive -> BridgeLink-ASL -> models/")
print("   Download cnn_landmark_best.pt and labels.json for the main demo")
print("   Optional: download sign_transformer_best.pt for the Transformer extension")
print("2. Go to Google Drive -> BridgeLink-ASL -> results/")
print("   Download cnn_metrics.json, CNN plots, and optional Transformer metrics")
print("3. Go to Google Drive -> BridgeLink-ASL -> vlm_eval_wlasl25_cnn/")
print("   Download wlasl25_cnn_hybrid_eval.jsonl and clips/ for VLM reranking")
print("=" * 60)


## What you just built

- **Data**: WLASL-100 filtered subset
- **Features**: MediaPipe Holistic -> 225-d landmark vectors x 32 frames per clip
- **Primary model**: temporal 1D CNN over MediaPipe landmark sequences
- **Main comparison**: CNN top-1/top-5 versus zero-shot VLM reranking over CNN top-5 candidates
- **Extra model**: optional 4-layer Transformer encoder for the attention/modern-methods rubric item
- **CNN training**: 50 epochs, AdamW, cosine schedule, label smoothing, gradient clipping, landmark augmentation

All outputs are in your Google Drive under `BridgeLink-ASL/`. They will survive if this Colab session dies.
